## **Ứng dụng thuật toán LightGBM trong dự đoán nguy cơ mắc bệnh tiểu đường**.   

### **Data processing** 

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
# Hỗ trợ chạy notebook từ repo root hoặc trực tiếp trong thư mục classification.
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)
REPO_ROOT = next((
    path for path in repo_candidates
    if (path / "classification" / "lightgbm_classification.py").is_file()
), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy repo root chứa classification/lightgbm_classification.py."
    )

repo_root_text = str(REPO_ROOT)
if repo_root_text not in sys.path:
    sys.path.insert(0, repo_root_text)

from classification.lightgbm_classification import LightGBMClassification
from classification.evaluation.run_diabetes_evaluation import (
    DEFAULT_OUTPUT_DIR,
    TARGET_COLUMN,
    build_diabetes_classifier,
    evaluate_diabetes_splits,
    split_diabetes_dataset,
)


In [2]:
# Đọc dữ liệu 
DATA_PATH = REPO_ROOT / "classification" / "data" / "diabetes_binary_health_indicators_BRFSS2015.csv"
df = pd.read_csv(DATA_PATH)
print ('Kích thước dataset', df.shape ) 

df.head ( 10 )

Kích thước dataset (253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
5,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,6.0,8.0
6,0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,0.0,14.0,0.0,0.0,9.0,6.0,7.0
7,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,0.0,0.0,1.0,0.0,11.0,4.0,4.0
8,1.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0
9,0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,2.0,0.0,0.0,0.0,1.0,8.0,4.0,3.0


In [3]:
# Kiểm tra kiểu dữ liệu từng feature  
df.dtypes  

Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object

In [4]:
# Dữ liệu thiếu 
df.isnull ( ).sum ( )

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [5]:
# Tách đặc trưng (X) và mục tiêu (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

# Giữ phân bố gốc; không dùng SMOTE trước khi chia để tránh rò rỉ dữ liệu.
print("Phân bố nhãn gốc:")
print(y.value_counts())

Phân bố nhãn gốc:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64


In [6]:
# Kiểm tra phân bố nhãn ban đầu; SMOTE sẽ chỉ áp dụng sau khi tách train/test.
print("Phân bố nhãn gốc:")
print(y.value_counts())

Phân bố nhãn gốc:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64


In [7]:

# Chia dữ liệu 80/20 có stratify để giữ tỷ lệ của lớp thiểu số.
X_train_original, X_test, y_train_original, y_test = split_diabetes_dataset(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Tach validation tu train; validation va test giu phan bo goc.
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train_original, y_train_original, test_size=0.2, random_state=42,
    stratify=y_train_original,
)

# SMOTENC chi ap dung tren tap fit, sau khi tach validation/test.
# BRFSS co nhieu bien nhi phan/thu bac; SMOTENC tranh noi suy ra category khong hop le.
continuous_columns = ["BMI", "MentHlth", "PhysHlth"]
categorical_indices = [
    X_fit.columns.get_loc(column)
    for column in X_fit.columns
    if column not in continuous_columns
]
smote = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=1.0,
    random_state=42,
)
X_train, y_train = smote.fit_resample(X_fit, y_fit)

print("Phan bo nhan train sau SMOTE:")
print(y_train.value_counts())

# Kiểm tra kích thước sau SMOTE
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Phan bo nhan train sau SMOTE:
Diabetes_binary
0.0    139733
1.0    139733
Name: count, dtype: int64
X_train: (279466, 21)
X_validation: (40589, 21)
X_test : (50736, 21)
y_train: (279466,)
y_test : (50736,)


### **Training model** 

In [8]:
# Reload module de notebook luon dung hyperparameter moi nhat sau khi sua file .py.
import importlib
import classification.evaluation.run_diabetes_evaluation as diabetes_evaluation
diabetes_evaluation = importlib.reload(diabetes_evaluation)

# Model regularized: cay nong hon, learning rate thap hon va L1/L2 manh hon.
model_lightGBM_cls = diabetes_evaluation.build_diabetes_classifier()
print({
    name: getattr(model_lightGBM_cls, name)
    for name in ("n_estimators", "learning_rate", "num_leaves", "max_depth",
                 "min_child_samples", "reg_alpha", "reg_lambda", "feature_fraction")
})

{'n_estimators': 1000, 'learning_rate': 0.015, 'num_leaves': 128, 'max_depth': 10, 'min_child_samples': 50, 'reg_alpha': 0.5, 'reg_lambda': 15.0, 'feature_fraction': 0.6}


In [9]:
# Dung nguong mac dinh 0.5 de so sanh truc tiep giua cac cau hinh model.
model_lightGBM_cls.threshold = 0.5
print(f'Threshold mac dinh: {model_lightGBM_cls.threshold:.2f}')

Threshold mac dinh: 0.50


In [10]:
model_lightGBM_cls.fit(X_train, y_train)

# Chon threshold tren validation: toi da F1, nhung accuracy khong thap hon 0.85.
validation_proba = model_lightGBM_cls.predict_proba(X_validation)[:, 1]
thresholds = np.arange(0.05, 0.96, 0.01)
MIN_VALIDATION_ACCURACY = 0.85

def threshold_metrics_at(y_true, probabilities, threshold):
    y_pred = (probabilities >= threshold).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    accuracy = np.mean(y_pred == y_true)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return accuracy, precision, recall, f1

candidate_metrics = [
    (threshold, *threshold_metrics_at(y_validation.to_numpy(), validation_proba, threshold))
    for threshold in thresholds
]
eligible = [item for item in candidate_metrics if item[1] >= MIN_VALIDATION_ACCURACY]
# Trong cac nguong giu accuracy >= 0.85, chon F1 cao nhat; tie-break bang precision.
best_threshold, validation_accuracy, validation_precision, validation_recall, validation_f1 = (
    max(eligible, key=lambda item: (item[4], item[2]))
    if eligible
    else max(candidate_metrics, key=lambda item: (item[4], item[1]))
)
model_lightGBM_cls.threshold = float(best_threshold)
print(
    f'Validation threshold={best_threshold:.2f}; accuracy={validation_accuracy:.3f}; '
    f'precision={validation_precision:.3f}; recall={validation_recall:.3f}; F1={validation_f1:.3f}'
)

Validation threshold=0.52; accuracy=0.851; precision=0.454; recall=0.356; F1=0.399


###   **Model Performance Evaluation** 

In [ ]:
# Helper chung tự tạo predict/proba một lần cho cả train và test.
# Mỗi split có artifact riêng và root manifest chỉ cập nhật khi cả hai thành công.
EVALUATION_OUTPUT_ROOT = DEFAULT_OUTPUT_DIR
evaluation_run = evaluate_diabetes_splits(
    model=model_lightGBM_cls,
    X_train=X_fit,
    X_test=X_test,
    y_train=y_fit,
    y_test=y_test,
    output_dir=EVALUATION_OUTPUT_ROOT,
)

evaluation_results = evaluation_run["split_results"]
PRIMARY_SPLIT = evaluation_run["primary_reporting_split"]
primary_evaluation_result = evaluation_results[PRIMARY_SPLIT]
pipeline_manifest_path = evaluation_run["pipeline_manifest_path"]
print(f"Đã lưu train evaluation tại: {EVALUATION_OUTPUT_ROOT / 'train'}")
print(f"Đã lưu test evaluation (primary) tại: {EVALUATION_OUTPUT_ROOT / 'test'}")
print(f"Root manifest: {pipeline_manifest_path}")
primary_evaluation_result["classification_report"]

In [ ]:
# Bảng so sánh train/test; cột test_primary là kết quả báo cáo chính.
COMPARISON_METRICS = ("accuracy", "precision", "recall", "f1_score", "roc_auc")
metrics_comparison = pd.DataFrame(
    {
        "train": [
            evaluation_results["train"]["metrics"][metric_name]
            for metric_name in COMPARISON_METRICS
        ],
        "test_primary": [
            evaluation_results[PRIMARY_SPLIT]["metrics"][metric_name]
            for metric_name in COMPARISON_METRICS
        ],
    },
    index=pd.Index(COMPARISON_METRICS, name="metric"),
)
metrics_comparison

,train,test_primary
metric,,
accuracy,0.854319,0.848175
precision,0.471110,0.443774
recall,0.371232,0.353940
f1_score,0.415249,0.393799
roc_auc,0.825225,0.815678
